In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/piotrszczypior/robustness
%cd robustness/

In [ ]:
import pandas as pd

df = pd.read_csv("results/resnet152_imagenet.csv")

print(len(df))

df.head()

In [ ]:
class_stats = df.groupby('synset').agg(
        accuracy=('is_correct', 'mean'),
        avg_confidence=('confidence', 'mean'),
        samples_count=('is_correct', 'count')
    ).reset_index()


top_classes = class_stats.sort_values(
        by=['accuracy', 'avg_confidence'], ascending=[False, False]
    ).head(10)

worst_classes = class_stats.sort_values(
        by=['accuracy', 'avg_confidence'], ascending=[True, False]
    ).head(10)

print(worst_classes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_class_vulnerability(df: pd.DataFrame, output_path: str = "resnet152_class_vulnerability_imagenet1k_val.png"):
    class_stats = df.groupby('synset').agg(
        accuracy=('is_correct', 'mean'),
        avg_confidence=('confidence', 'mean')
    ).reset_index()

    plt.figure(figsize=(10, 8))
    
    sns.scatterplot(
        data=class_stats, 
        x='accuracy', 
        y='avg_confidence', 
        alpha=0.6, 
        edgecolor=None,
        color='blue'
    )
    
    plt.plot([0, 1], [0, 1], 'k--', label='Acc = Conf')
    
    plt.title("ResNet-152: Accuracy vs Confidence (ImageNet-1K val)", fontsize=14)
    plt.xlabel("Accuracy", fontsize=12)
    plt.ylabel("Confidence", fontsize=12)
    plt.xlim(-0.05, 1.05)
    plt.ylim(-0.05, 1.05)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


plot_class_vulnerability(df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_worst_classes_bar(df: pd.DataFrame, output_path: str = "worst_classes_bar.png"):
    class_stats = df.groupby('synset').agg(
        accuracy=('is_correct', 'mean')
    ).reset_index()
    
    worst_classes = class_stats.sort_values(by='accuracy', ascending=True).head(15)
    
    plt.figure(figsize=(10, 8))
    
    sns.barplot(
        data=worst_classes, 
        x='accuracy', 
        y='synset',
    )
    
    plt.title("ResNet-152: top 15 worst performing classes - ImageNet-1K val", fontsize=14)
    plt.xlabel("Accuracy", fontsize=12)
    plt.ylabel("Synset", fontsize=12)
    
    plt.xlim(0, 1)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()


plot_worst_classes_bar(df)